In [2]:
from dotenv import load_dotenv
from agents import Agent,Runner,trace,function_tool
import asyncio
import os

load_dotenv(override=True)

True

In [1]:
def scan_test_print():
    print()

In [3]:
instructions1 = "You are a sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails."

instructions2 = "You are a humorous, engaging sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response."

instructions3 = "You are a busy sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails."

sales_agent1 = Agent(
        name="Professional Sales Agent",
        instructions=instructions1,
        model="gpt-4o-mini"
)

sales_agent2 = Agent(
        name="Engaging Sales Agent",
        instructions=instructions2,
        model="gpt-4o-mini"
)

sales_agent3 = Agent(
        name="Busy Sales Agent",
        instructions=instructions3,
        model="gpt-4o-mini"
)

In [7]:

from openai.types.responses import ResponseTextDeltaEvent

result = Runner.run_streamed(sales_agent1, input="Write a cold sales email")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Subject: Streamline Your SOC 2 Compliance with ComplAI

Hi [Recipient's Name],

I hope this message finds you well. I'm [Your Name], a representative from ComplAI, where we specialize in simplifying the SOC 2 compliance process for businesses like yours.

Maintaining SOC 2 compliance can be a daunting task, often involving complex documentation and ongoing audits. Our AI-powered platform is designed to streamline these efforts, automating key processes to save you time and reduce stress. 

Here’s what we offer:

- **Automated Documentation**: Simplify the creation and management of compliance documents.
- **Continuous Monitoring**: Stay ahead of compliance requirements with real-time alerts and insights.
- **Audit Preparation**: Ensure you are fully prepared for audits with our comprehensive toolset.

Many companies have already seen a significant reduction in compliance-related workload, allowing them to focus on their core business operations.

I would love to discuss how ComplAI can

In [8]:
message = "Write a cold sales email"

with trace("Parallel cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1,message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )
outputs = [result.final_output for result in results]

for output in outputs:
    print(output + "\n\n")

Subject: Streamline Your SOC 2 Compliance with ComplAI

Dear [Recipient's Name],

I hope this message finds you well. My name is [Your Name], and I’m reaching out from ComplAI, where we specialize in helping organizations navigate the complexities of SOC 2 compliance effortlessly.

In today's fast-paced digital environment, demonstrating your commitment to security and regulatory standards is critical not only for compliance but also for building trust with your clients. Unfortunately, the traditional methods of preparing for audits are often time-consuming and prone to human error.

That’s where ComplAI comes in. Our AI-powered SaaS tool automates your compliance processes, streamlining the entire journey from preparation to audit. By leveraging our solution, you can expect:

- **Simplified Documentation**: Create and manage necessary documentation efficiently.
- **Continuous Monitoring**: Ensure ongoing compliance with real-time updates and alerts.
- **Insightful Analytics**: Identif

In [9]:
sales_picker = Agent(
    name="sales_picker",
    instructions="You pick the best cold sales email from the given options. \
Imagine you are a customer and pick the one you are most likely to respond to. \
Do not give an explanation; reply with the selected email only.",
    model="gpt-4o-mini"
)

In [10]:
message = "Write a cold sales email"

with trace('Selection from sales people'):
    results = await asyncio.gather(
        Runner.run(sales_agent1,message),
        Runner.run(sales_agent2,message),
        Runner.run(sales_agent3,message)

    )

    outputs =[result.final_output for result in results]
    emails = "Cold sales emails:\n \n Email:\n\n ".join(outputs)

    best = await Runner.run(sales_picker,emails)

    print(f"est sales email :\n {best.final_output}")



est sales email :
 Subject: Let’s Make Your SOC 2 Journey as Smooth as Butter! 🧈

Hi [Recipient's Name],

Ever tried to dance the Macarena with an elephant on your back? That’s what navigating SOC 2 compliance can feel like! 🐘💃

At ComplAI, we believe your compliance journey should be less circus and more smooth sailing. Our AI-powered tool simplifies SOC 2 compliance, so you can spend less time juggling paperwork and more time perfecting your dance moves (or, y’know, running your business).

✨ Here’s how we can help you:

- **Automated Documentation**: Wave goodbye to endless spreadsheets. Our tool does the heavy lifting so you can kick back with a cup of coffee. ☕
- **Audit Prep Made Easy**: We turn your audits from a scary bear into a cuddly teddy! 🐻✨
- **Real-Time Monitoring**: Keep your systems in check without breaking a sweat. Who doesn’t want to be the calm in the storm?

Let’s chat! I promise, no elephant dance moves are required. Just a quick call to see how we can help you c

In [11]:
#Now Handle tool calls
sales_agent1 = Agent(
        name="Professional Sales Agent",
        instructions=instructions1,
        model="gpt-4o-mini",
)

sales_agent2 = Agent(
        name="Engaging Sales Agent",
        instructions=instructions2,
        model="gpt-4o-mini",## ##
)

sales_agent3 = Agent(
        name="Busy Sales Agent",
        instructions=instructions3,
        model="gpt-4o-mini",
)

In [12]:
sales_agent1

Agent(name='Professional Sales Agent', instructions='You are a sales agent working for ComplAI, a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. You write professional, serious cold emails.', handoff_description=None, handoffs=[], model='gpt-4o-mini', model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, metadata=None, store=None, include_usage=None, extra_query=None, extra_body=None, extra_headers=None), tools=[], mcp_servers=[], mcp_config={}, input_guardrails=[], output_guardrails=[], output_type=None, hooks=None, tool_use_behavior='run_llm_again', reset_tool_choice=True)

In [13]:
@function_tool
def print_email_statemment(body:str):
    """This prints out the statements that are given by the agents."""
    print(body)

print_email_statemment

FunctionTool(name='print_email_statemment', description='This prints out the statements that are given by the agents.', params_json_schema={'properties': {'body': {'title': 'Body', 'type': 'string'}}, 'required': ['body'], 'title': 'print_email_statemment_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x112552160>, strict_json_schema=True, is_enabled=True)

Agent(name='Professional Sales Agent', instructions='You are a sales agent working for ComplAI, a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. You write professional, serious cold emails.', handoff_description=None, handoffs=[], model='gpt-4o-mini', model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, metadata=None, store=None, include_usage=None, extra_query=None, extra_body=None, extra_headers=None), tools=[], mcp_servers=[], mcp_config={}, input_guardrails=[], output_guardrails=[], output_type=None, hooks=None, tool_use_behavior='run_llm_again', reset_tool_choice=True)
Agent(name='Engaging Sales Agent', instructions='You are a humorous, engaging sales agent working for ComplAI, a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI

In [18]:
from agents import Agent,Runner,tool,function_tool
description = "Write a cold sales email"
tool =[]
sales_agents = [sales_agent1, sales_agent2, sales_agent3]
# Now covert all agents as tools
for i,agent in enumerate(sales_agents):
    agent_tool = agent.as_tool(tool_name=f"sales_agent{i}",tool_description=description)
    tool.append(agent_tool)

    

tool.append(print_email_statemment)
tool

[FunctionTool(name='sales_agent0', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent0_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x11375c400>, strict_json_schema=True, is_enabled=True),
 FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x11375cea0>, strict_json_schema=True, is_enabled=True),
 FunctionTool(name='sales_agent2', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required'

In [19]:
# Improved instructions thanks to student Guillermo F.

instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Use the print_email_statemment tool to print the best email (and only the best email) to the user.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must print ONE email using the print_email_statemment tool — never more than one.
"""


sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tool, model="gpt-4o-mini")

message = "Send a cold sales email addressed to 'Dear CEO'"

with trace("Sales manager"):
    result= await Runner.run(sales_manager,message)


Subject: Is Your Compliance as Cozy as Your Favorite Sweater? 

Hey [CEO's Name],

I hope this email finds you well and not buried under a mountain of audit checklists! 😅 

As a fellow enthusiast of all things techy (and cozy), I couldn’t help but notice that achieving SOC2 compliance might be feeling a bit like trying to solve a Rubik's Cube blindfolded. Spoiler alert: it doesn't have to be that hard!

At ComplAI, we’ve brewed up a secret sauce of AI-powered tools to turn those complex regulations into something you can actually manage—no magic wands required! Think of us as your new compliance BFF, always there to help you keep those audits from turning into stress-fests.

Got 15 minutes this week for a quick chat? I promise I’ll keep the awkward silences to a minimum! 

Warmest wishes (and no more compliance headaches),

[Your Name]  
Sales Manager, ComplAI  
[Your Phone Number]  
[Your Email]  

P.S. If you’re a fan of dad jokes, I’ve got a few compliance-related ones up my sleeve!

In [21]:
result.final_output

"The best cold sales email is:\n\n---\n\n**Subject: Is Your Compliance as Cozy as Your Favorite Sweater?**\n\nHey [CEO's Name],\n\nI hope this email finds you well and not buried under a mountain of audit checklists! 😅 \n\nAs a fellow enthusiast of all things techy (and cozy), I couldn’t help but notice that achieving SOC2 compliance might be feeling a bit like trying to solve a Rubik's Cube blindfolded. Spoiler alert: it doesn't have to be that hard!\n\nAt ComplAI, we’ve brewed up a secret sauce of AI-powered tools to turn those complex regulations into something you can actually manage—no magic wands required! Think of us as your new compliance BFF, always there to help you keep those audits from turning into stress-fests.\n\nGot 15 minutes this week for a quick chat? I promise I’ll keep the awkward silences to a minimum! \n\nWarmest wishes (and no more compliance headaches),\n\n[Your Name]  \nSales Manager, ComplAI  \n[Your Phone Number]  \n[Your Email]  \n\nP.S. If you’re a fan of 

In [33]:
## Revision, so now we will be doing the things that OpenA SDK is all about. OpenA SDK is all about
from agents import Agent,Runner,function_tool

marketing_agent1 = Agent(name="AlphaAgent",instructions="Your target is to write an email to Gen Alpha kids. You can look for the things that Gen Alpha people speak like, name templates that they see around the terms that they use and those kinds of things. So you need to target them and you need to write a mail to them and you need to make them willing to buy our product"
,model = "gpt-4o-mini")


marketing_agent2 = Agent(name="ZAgent",instructions="Your target is the Gen Z kids, write an email and what they spoke, what they contemplated, and what they find funny, so in their terms you need to sell our product."
,model = "gpt-4o-mini")

marketing_agent3 = Agent(name="Others",instructions="Your target is to write a formal mail for 90's or 80's kids or like the others basically. Like not the Gen Z kids or not the Gen R kids. Use a formal template, address them, say how the product is the best."
,model = "gpt-4o-mini")
publish_agent = Agent(name="pub_agent",instructions="Your job is to add Terms and conditions apply and other conditions that may apply that come along with our milk product so that we won't get any legal issues.",model = "gpt-4o-mini"
,handoff_description="checks for legal issues and prints it")

marketing_agents = [marketing_agent1,marketing_agent2,marketing_agent3]



description = "the milk that we are selling. This milk particularly has a higher shelf value, higher shelf life, and we have also increased the protein content of the milk artificially. Do mention that because of this, it might taste a little different from the regular milk. Like some, we would be getting a sour taste. Age 12"
tools=[]
for i,agent in  enumerate(marketing_agents):
    tool_agent = agent.as_tool(tool_name=agent.name,tool_description=description)

    tools.append(tool_agent)

tools.append(print_email_statemment)



market_head = Agent(name="Head",instructions="""You will be provided with the output of all the agents. 
You need to select which one you should send the user and delegate it to the `pub_agent` so it prints the email, 
but check with acc. to the age group specified, and also, if something is not good, you can ask the other agent to redo the work, and then you need to accept it.
Rules:
Age <= 13 → AlphaAgent
Age 14-28 → ZAgent
Age > 28 → Others
Handoff for Sending: Pass ONLY the winning email draft to the 'pub_agent' agent"""
,model = "gpt-4o-mini",tools=tools,handoffs=[publish_agent])






In [34]:
message = "Send a mail to a 12-year-old."

with trace("Marketing manager Aavin"):
    result = await Runner.run(market_head, message)
print(result)

Subject: 🌟 Hey Superstars! Discover Our Amazing Milk! 🥛✨

Hey there, Future Legends! 🌈

We hope you're ready to embark on another cool adventure! 🚀 Today, let’s talk about something super awesome—our special milk! 🥛✨

🌟 **What Makes It Special?**
- **Longer Shelf Life:** No more worrying about your milk going bad! Our milk stays fresh and cool way longer, so you can enjoy it whenever you want! 🕒😄
- **Power-Packed Protein:** Packed with more protein than regular milk, it’s perfect for fueling your after-school adventures! Whether you’re playing games, creating epic art, or conquering homework, this milk has your back! 💪🎨📚

**A Little Tip:** Some taste explorers say our milk has a unique flavor that might be a bit sour. But who doesn’t love trying something new? 🤔✨

🌈 **Join the Milk Movement!**
We know you’re always on the lookout for the coolest stuff. If you’re ready to try something different and delicious, grab a carton of our special milk and let us know what you think! You can eve